# Creation of the Dataset for the prediction task
- We want to create a dataset where we create some features that could be useful for the task of classifying wether a cyclist is going to place in the top20 or not (binary classification).
- As we noticed in the previous tasks, the history of a cyclist is an important factor to have in order to obtain knowledge and we think that it is going also to be crucial for this kind of classification.
- We want to create features that represent the history of the cyclists of his "previous races". For previous races we mean the races made by the cyclist before the race we are taking into consideration, for example:
    - Consider a cyclist that made 3 races, one for each year (2000,2001,2002), when we create the features for the cyclist in the year 2002 we will take into consideration the races made in 2000 and 2001, when we consider the race made in 2000 we do not have this kind of information and the feature will result NaN.

# Features created
- <span style="color: red;">avg_position</span>: The average position of a cyclist in the previous races
- <span style="color: red;">avg_position_x</span>: The average position of a cyclist in the previous x races, when x races are not available, just consider the maximum possible amount, but still need a minumum given by x-1.
- <span style="color: red;">races_amount</span>: The amount of races made by a cyclist
- <span style="color: red;">performance_wma</span>: moving average of the position of previous races, it considers a window of 5 previous races
- <span style="color: red;">top20 consistency</span>: it is the amount of times a cyclist ended up in top20 in the previous races
- <span style="color: red;">races_dby_consistency</span>: top20 consistency divided by races_amount, to avoid division by 0, top20_series is set to 1 when is 0
- <span style="color: red;">prev_position_x</span>: the position of the cyclist x races ago.


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import gzip
import numpy as np

save = True
save2 = True

In [3]:
# Take the merged dataset cleaned from null values and anomalies
with gzip.open('../filtered_clean_merged_df.pkl', 'r') as file:
    filtered_clean_dataset = pickle.load(file)

In [4]:
# Sort the dataset by date
df_ordered_by_date = filtered_clean_dataset.sort_values(['date'])

In [5]:
df_ordered_by_date_races = df_ordered_by_date.groupby(['race_id']).size().reset_index(name='participants')
df_ordered_by_date = pd.merge(df_ordered_by_date, df_ordered_by_date_races)

In [6]:
races = df_ordered_by_date.groupby('cyclist')['position']

# Create avg_position
for group_name,group_data in races:
    avg_positions = group_data.expanding().mean().shift()
    df_ordered_by_date.loc[group_data.index, 'avg_position'] = avg_positions


In [10]:
# Create avg_position_1
for group_name,group_data in races:
    avg_positions_1 = group_data.rolling(window=1, min_periods=1).mean().shift()
    df_ordered_by_date.loc[group_data.index, 'avg_position_1'] = avg_positions_1

In [11]:
# Create avg_position_2
for group_name,group_data in races:
    avg_positions_2 = group_data.rolling(window=2, min_periods=1).mean().shift()
    df_ordered_by_date.loc[group_data.index, 'avg_position_2'] = avg_positions_2

In [12]:
# Create avg_position_3
for group_name,group_data in races:
    avg_positions_3 = group_data.rolling(window=3, min_periods=2).mean().shift()
    df_ordered_by_date.loc[group_data.index, 'avg_position_3'] = avg_positions_3

In [13]:
# Create avg_position_4
for group_name,group_data in races:
    avg_positions_4 = group_data.rolling(window=4, min_periods=3).mean().shift()
    df_ordered_by_date.loc[group_data.index, 'avg_position_4'] = avg_positions_4

In [14]:
# Create races_amount
for group_name,group_data in races:
    races_amount = group_data.expanding().count().shift()
    df_ordered_by_date.loc[group_data.index, 'races_amount'] = races_amount

In [15]:
# Weighted moving average
for group_name, group_data in races:
    performance_wma = group_data.ewm(span=5, adjust=False).mean().shift()
    df_ordered_by_date.loc[group_data.index, 'performance_wma'] = performance_wma

In [16]:
# top20 consistency
for group_name, group_data in races:
    top20_series = (group_data <= 20).expanding().sum().shift()
    df_ordered_by_date.loc[group_data.index, 'top20_consistency'] = top20_series

In [17]:
# top20 consistency divided by races_amount, to avoid division by 0, top20_series is set to 1 when is 0
for group_name, group_data in races:
    races_amount = group_data.expanding().count().shift()
    top20_series = (group_data <= 20).expanding().sum().shift()
    top20_series[top20_series == 0] = 1
    races_dby_consistency = races_amount/top20_series
    df_ordered_by_date.loc[group_data.index, 'races_dby_consistency'] = races_dby_consistency

In [18]:
# Create prev_position_1
for group_name,group_data in races:
    prev_position_1 = group_data.shift(1)
    df_ordered_by_date.loc[group_data.index, 'prev_position_1'] = prev_position_1

In [19]:
# Create prev_position_2
for group_name,group_data in races:
    prev_position_2 = group_data.shift(2)
    df_ordered_by_date.loc[group_data.index, 'prev_position_2'] = prev_position_2

In [20]:
# Create prev_position_3
for group_name,group_data in races:
    prev_position_3 = group_data.shift(3)
    df_ordered_by_date.loc[group_data.index, 'prev_position_3'] = prev_position_3

In [21]:
# Create prev_position_4
for group_name,group_data in races:
    prev_position_4 = group_data.shift(4)
    df_ordered_by_date.loc[group_data.index, 'prev_position_4'] = prev_position_4

We consider as test set the races made from 2022 onwards.

In [22]:
# Create training set: races until 2021
train_set = df_ordered_by_date[df_ordered_by_date['year'] <= 2021]

# Create test set: races from 2022 onwards
test_set = df_ordered_by_date[df_ordered_by_date['year'] > 2021]

train_set['top20'] = (train_set['position'] <= 20).astype(int)
test_set['top20'] = (test_set['position'] <= 20).astype(int)

C:\Users\Fede\AppData\Local\Temp\ipykernel_14544\2987656537.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_set['top20'] = (train_set['position'] <= 20).astype(int)
C:\Users\Fede\AppData\Local\Temp\ipykernel_14544\2987656537.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_set['top20'] = (test_set['position'] <= 20).astype(int)


In [23]:
if save:
    with gzip.open('train_set.pkl', 'wb') as file:
        pickle.dump(train_set, file)
    with gzip.open('test_set.pkl', 'wb') as file:
        pickle.dump(test_set, file)
if save2:
    with gzip.open('prediction_dataset.pkl', 'wb') as file:
        pickle.dump(df_ordered_by_date, file)